In [ ]:
from typing import TypedDict, Literal

from langchain_core.runnables import RunnableConfig
from langgraph.managed import  RemainingSteps
from loguru import  logger
from langgraph.graph import StateGraph,START,END
from rich import print

# 1. 状態を宣言
class OverAllState(TypedDict):
    remaining_steps:RemainingSteps

# 2. ループノードを宣言
def loop_node(state:OverAllState,config:RunnableConfig) -> OverAllState:
    cur_step = config["metadata"]["langgraph_step"]
    # print(f"config --> {config}")
    remaining_steps = state["remaining_steps"]
    logger.info("loop_node,cur_step:{},remaining_step:{}",cur_step,remaining_steps)

#3. ルーターを追加
def router(state:OverAllState) -> Literal["loop_node",END]:
    remaining_steps = state["remaining_steps"]
    if remaining_steps < 3:
        logger.info("現在利用可能なスーパーステップ:{}、残り3ステップ未満のためループを終了します",remaining_steps)
        return END
    return "loop_node"

#4. グラフを構築
builder = StateGraph(state_schema=OverAllState)

builder.add_node("loop_node",loop_node)

builder.add_edge(START,"loop_node") 
builder.add_conditional_edges("loop_node",router)

graph = builder.compile()
graph.invoke({},config={"recursion_limit": 20})


In [ ]:
from IPython.display import display
display(graph)